# Staged RL Training for Qwen2.5-VL

This notebook copies the uploaded code dataset into `/kaggle/working`, installs dependencies, and runs one explicit training phase.

In [ ]:
import os, shutil, subprocess, glob, json, tarfile

WORKDIR = "/kaggle/working/RL_GSPO_Qwen2.5VLM"
VENV_DIR = "/tmp/rl-gspo-venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"

matches = []
for root, _, files in os.walk("/kaggle/input"):
    if "rl_gspo_qwen2_5vlm_test3.py" in files:
        matches.append(root)
if not matches:
    raise FileNotFoundError(f"Could not find attached code dataset. Visible inputs: {glob.glob('/kaggle/input/*')}")

if len(matches) > 1:
    raise RuntimeError(f"Ambiguous code dataset attachment. Matches={matches}. Visible inputs: {glob.glob('/kaggle/input/*')}")

src = matches[0]
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(src, WORKDIR)
for archive_name in ("staged_rl.tar", "tests.tar"):
    archive_path = os.path.join(WORKDIR, archive_name)
    if os.path.exists(archive_path):
        with tarfile.open(archive_path, "r") as tf:
            tf.extractall(WORKDIR)
        os.remove(archive_path)
        print("Extracted", archive_name)
for folder_name in ("staged_rl", "tests"):
    nested_path = os.path.join(WORKDIR, folder_name, folder_name)
    target_path = os.path.join(WORKDIR, folder_name)
    if os.path.isdir(nested_path):
        temp_path = f"{target_path}_flat"
        if os.path.exists(temp_path):
            shutil.rmtree(temp_path)
        shutil.move(nested_path, temp_path)
        shutil.rmtree(target_path)
        shutil.move(temp_path, target_path)
        print("Flattened", folder_name)
print("Using code dataset", src)
print("Copied code to", WORKDIR)
print(sorted(os.listdir(WORKDIR)))

In [ ]:
subprocess.run(["python3", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["python3", "-m", "uv", "venv", "--seed", "--clear", VENV_DIR], check=True)
install_commands = [
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--upgrade", "pip", "setuptools", "wheel"],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "numpy==1.26.4",
        "scipy==1.15.3",
        "scikit-learn==1.6.1",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "pillow",
        "packaging",
        "safetensors",
        "torchvision",
        "bitsandbytes",
        "xformers",
        "huggingface_hub>=0.34.0",
        "datasets==4.3.0",
        "transformers==4.56.2",
        "trl==0.22.2",
        "unsloth",
        "modelscope",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "vllm==0.10.2",
        "--extra-index-url",
        "https://wheels.vllm.ai/0.10.2/",
    ],
]
for command in install_commands:
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=WORKDIR)
print("Venv ready:", VENV_PYTHON)

In [ ]:
compat_script = """
import numpy
import scipy
import sklearn
import vllm
from vllm.sampling_params import GuidedDecodingParams
from trl import GRPOTrainer
print('numpy version:', numpy.__version__)
print('scipy version:', scipy.__version__)
print('sklearn version:', sklearn.__version__)
print('vllm version:', vllm.__version__)
print('compat imports OK')
"""
subprocess.run([VENV_PYTHON, "-c", compat_script], check=True, cwd=WORKDIR)

In [ ]:
HARDWARE_PROFILE = "kaggle_t4"
TRAIN_SPLIT = "test"
EVAL_SPLIT = "testmini"
MAX_EVAL_EXAMPLES_PER_SUBSET = 4
PHASE_RUNS = [
    ("phase_a", None),
    ("phase_b", "best_structure"),
    ("phase_c", "best_composite"),
]

env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["UNSLOTH_USE_MODELSCOPE"] = "1"

for PHASE, RESUME in PHASE_RUNS:
    cmd = [
        VENV_PYTHON,
        "rl_gspo_qwen2_5vlm_test3.py",
        "--phase", PHASE,
        "--hardware-profile", HARDWARE_PROFILE,
        "--train-split", TRAIN_SPLIT,
        "--eval-split", EVAL_SPLIT,
        "--max-eval-examples-per-subset", str(MAX_EVAL_EXAMPLES_PER_SUBSET),
    ]
    if RESUME:
        cmd.extend(["--resume", RESUME])

    phase_output_dir = os.path.join(WORKDIR, "outputs_staged", PHASE)
    os.makedirs(phase_output_dir, exist_ok=True)
    train_log_path = os.path.join(phase_output_dir, "train_subprocess.log")
    print("Running:", " ".join(cmd))
    print("Subprocess log:", train_log_path)
    with open(train_log_path, "w", encoding="utf-8") as log_file:
        completed = subprocess.run(
            cmd,
            check=False,
            cwd=WORKDIR,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )
    if completed.returncode != 0:
        print(f"Phase {PHASE} failed with return code {completed.returncode}. Last 200 log lines:")
        with open(train_log_path, "r", encoding="utf-8") as log_file:
            tail_lines = log_file.readlines()[-200:]
        print("".join(tail_lines))
        raise subprocess.CalledProcessError(completed.returncode, cmd)
    print(f"Phase {PHASE} completed successfully.")


In [ ]:
outputs_root = os.path.join(WORKDIR, "outputs_staged")
collected = []
for root, _, files in os.walk(outputs_root):
    for file_name in files:
        collected.append(os.path.join(root, file_name))
for path in sorted(collected)[-50:]:
    print(path)